![SIMEM_logo_1.png](../../assets/SIMEM_logo_1.png)


[SiMEM](https://www.simem.co/)

## Ejemplo: Consulta y Análisis de Generación Distribuida

Este notebook muestra cómo consultar y analizar la información del archivo de **Generación Distribuida (kWh)** desde la API SIMEM.

### Descripción del contenido

El documento se organiza en las siguientes secciones:

1. **Consulta de datos mediante el API SIMEM**: Se utiliza la librería `pydataxm` para consultar:
   - **Generación Real (GReal)** — variable con versiones, usando `VariableSIMEM`
   - **Listado de Plantas** y **Plantas en Pruebas** — datasets `0BFC9D` y `7F18B1`
   - **Áreas Operativas** — dataset `841808`
   - **Agentes** — dataset `972263`

2. **Integración de datos**: Se unifican los listados de plantas, se cruzan con la generación real horaria y se enriquecen con nombres de áreas operativas y agentes.

3. **Análisis de Generación Distribuida**: Se filtra por `TipoClasificacion == 'GENERADOR DISTRIBUIDO'` y se construyen gráficos de:
   - Generación mensual por tipo de generación
   - Generación total por área operativa
   - Top 10 agentes por generación
   - Evolución diaria por tipo de generación
   - Distribución por área operativa y tipo de generación

4. **Participación en el total del sistema**: Se compara la generación distribuida contra la generación total (sin filtros) para evaluar su representatividad mediante gráficos de evolución diaria, porcentaje de participación y resumen mensual.

### Índice

1. [Importación de la clase VariableSIMEM](#section1)
2. [Consultar el listado de variables disponibles](#section2)
3. [Buscar el código de una variable](#section3)
4. [Consultar datos de la variable](#section4)

<a id='section1'></a>
### Importación de la clase VariableSIMEM

En la siguiente celda se importa el objeto VariableSIMEM, que contiene las funciones necesarias para cumplir los objetivos anteriormente mencionados.

In [73]:
import sys

# !{sys.executable} -m pip install pydataxm
from pydataxm.pydatasimem import ReadSIMEM, VariableSIMEM # Importa la libreria que fue instalada con pip install pydataxm
import pandas as pd
import plotly.graph_objects as go

<a id='section2'></a>
### Consultar el listado de variables disponibles dentro de la clase

En la siguiente celda se muestra como consultar el listado de variables con el fin de buscar el código de la variable deseada.

In [74]:
lisatado_variables = VariableSIMEM.get_collection() # Obtiene el listado de variables que se pueden usar dentro de VariableSIMEM
lisatado_variables.head()                           # Imprime los primeros 5 registros del listado de variables

,CodigoVariable,Nombre,Dimensiones
0,AportesHidricosEnergia,Aportes hidricos de las series hidrologicas ex...,"[CodigoSerieHidrologica, RegionHidrologica, Fe..."
1,AportesHidricosMasa,Aportes hidricos de las series hidrologicas,"[CodigoSerieHidrologica, RegionHidrologica, Fe..."
2,AportesHidricosMasaPSS95,Aportes hidricos para un 95% de Probabilidad d...,"[CodigoSerieHidrologica, RegionHidrologica, Fe..."
3,BanderaAutorizacionDesvFueraBanda,Banderas de autorizacion a desviarse,"[CodigoPlanta, Version, FechaHora]"
4,BanderaArranqueGIdeal,Bandera de arranque de un recurso termico,"[CodigoPlanta, Version, FechaHora]"


<a id='section3'></a>
### Buscar el código de una variable de interes Desde la API de SIMEM

En la siguiente celda se muestra como consultar el código de una variable a través de su nombre

In [79]:
nombre_variable = 'generacion' # Nombre de la variable que se desea buscar
cod_variable = lisatado_variables.query("Nombre.str.lower().str.contains(@nombre_variable.lower())")['CodigoVariable'].values # Busca el nombre dentro del listado de variables
print(cod_variable)                 # Imprime los códigos de las variables que coinciden con el nombre

['BanderaGidealInflexibleNoGenSeguridad'
 'BanderaGidealInflexibleGenSeguridad' 'DeltaGProg' 'DispCom'
 'EnergiaReferSTNnacional' 'GeneracionRealEstimada' 'GeneracionSeg'
 'GIdeal' 'GIdealInflexible' 'GIdealInt' 'GIdealNalFlexible' 'GIdealNal'
 'GIdealTIEFlexible' 'GIdealTie' 'GInflexiblesCandidatas' 'GProg'
 'GProgDespacho' 'GProgGrupoRedespacho' 'GProgRedespacho' 'GReal'
 'MgDeclaracionRespOEFCombLiquidoCompraGas' 'MgEnergiaVendEmbal_Ajustada'
 'MgGeneracionSegFueraMerito' 'PPBOGReal']


Una vez obtenido el código de la variable de interés, se debe inicializar una instancia de la clase VariableSIMEM con este y dos fechas de interés, para poder obtener la información de la variable en la versión deseada.
Hay 2 formas de solicitar la versión:
- Versión específica.
- Versiones en orden.

Versión en orden : Se debe pasar como parámetro numérico el orden que se desea. El orden es manejado de la siguiente forma:
- 0 -> Última versión.
- -1 -> Penúltima versión.
- -2 -> Antepenúltima versión.
- ....... y así sucesivamente.

In [108]:
cod_variable = 'GReal'      # Código de la variable deseada
fecha_inicio = '2026-01-01' # Fecha inicial de los datos
fecha_fin = '2026-05-05'    # Fecha final de los datos
version = 0                 # Versión de los datos

variable_ultima_version = VariableSIMEM(cod_variable, fecha_inicio, fecha_fin, version) # Se inicializa el objeto VariableSIMEM con última versión

In [109]:
data_ultima_version = variable_ultima_version.get_data() # Obtiene los datos de la variable con última versión
data_ultima_version

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Generación real"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0010509490966796875
Extraccion de registros: 148.73902320861816
End of data extracting process
****************************************************************************************************


,,,CodigoVariable,CodigoDuracion,UnidadMedida,CodigoSICAgente,Valor
FechaHora,Version,CodigoPlanta,,,,,
2026-01-26T09:00:00,TX3,TSR1,GReal,PT1H,kWh,EPMG,0.00
2026-01-26T04:00:00,TX3,COE1,GReal,PT1H,kWh,ENGG,1003.95
2026-01-26T10:00:00,TX3,3EV3,GReal,PT1H,kWh,EPSG,8513.30
2026-01-26T13:00:00,TX3,TBQ4,GReal,PT1H,kWh,TBSG,0.00
2026-01-26T21:00:00,TX3,3HG9,GReal,PT1H,kWh,ERRG,0.00
...,...,...,...,...,...,...,...
2026-05-02T21:00:00,TX1,5MAG,GReal,PT1H,kWh,OTAG,0.00
2026-05-02T06:00:00,TX1,5ISN,GReal,PT1H,kWh,GCYG,14.67
2026-05-02T12:00:00,TX1,3BZB,GReal,PT1H,kWh,EPSG,5842.10


In [110]:
data_ultima_version.reset_index(inplace=True) # Resetea el índice para que la fecha sea una columna normal
data_ultima_version.head()

,FechaHora,Version,CodigoPlanta,CodigoVariable,CodigoDuracion,UnidadMedida,CodigoSICAgente,Valor
0,2026-01-26T09:00:00,TX3,TSR1,GReal,PT1H,kWh,EPMG,0.00
1,2026-01-26T04:00:00,TX3,COE1,GReal,PT1H,kWh,ENGG,1003.95
2,2026-01-26T10:00:00,TX3,3EV3,GReal,PT1H,kWh,EPSG,8513.30
3,2026-01-26T13:00:00,TX3,TBQ4,GReal,PT1H,kWh,TBSG,0.00
4,2026-01-26T21:00:00,TX3,3HG9,GReal,PT1H,kWh,ERRG,0.00


In [97]:
dataset_id = '0BFC9D' # Código del dataset que se desea consultar, 
                      #  este código se obtiene del listado de variables, en la columna "DatasetID" y obtiene el Listado de plantas en operación
fecha_inicio = '2026-01-01'
fecha_fin = '2026-05-05'
simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
df_plantas = simem.main()
df_plantas.head(10)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Parámetros técnicos de las plantas de generación"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0010073184967041016
Extraccion de registros: 10.652338027954102
End of data extracting process
****************************************************************************************************


,Fecha,CodigoDuracion,FechaPublicacion,CodigoPlanta,NombrePlanta,CodigoSICAgente,CapEfectivaNeta,FPO,CodigoSubAreaOperativa,CodigoAreaOperativa,TipoDespachoRecurso,TipoClasificacion,TipoGeneracion
0,2026-01-19,P1D,2026-01-19,3IZ6,PARQUE SOLAR LA UNION,SLUG,100000.0,2024-06-19,Are0021,Are0127,Despachado Centralmente,GENERADOR,Solar
1,2026-01-19,P1D,2026-01-19,GYPO,GUAYEPO,ENDG,370000.0,2024-11-30,Are0016,Are0127,Despachado Centralmente,GENERADOR,Solar
2,2026-01-19,P1D,2026-01-19,3INX,CARACOLI I,SDCG,50000.0,2024-12-22,Are0016,Are0127,Despachado Centralmente,GENERADOR,Solar
3,2026-01-19,P1D,2026-01-19,3HF5,FUNDACION,ENDG,100000.0,2024-06-24,Are0025,Are0127,Despachado Centralmente,GENERADOR,Solar
4,2026-01-19,P1D,2026-01-19,EPFV,EL PASO,ENDG,68000.0,2024-03-23,Are0025,Are0127,Despachado Centralmente,GENERADOR,Solar
5,2026-01-19,P1D,2026-01-19,3DDT,LATAM SOLAR LA LOMA,ENDG,150000.0,2024-06-24,Are0025,Are0127,Despachado Centralmente,GENERADOR,Solar
6,2026-01-19,P1D,2026-01-19,MATA,LA MATA,SLMG,80000.0,2024-06-27,Are0029,Are0012,Despachado Centralmente,GENERADOR,Solar
7,2026-01-19,P1D,2026-01-19,3IQA,SUNNORTE,GNSG,35000.0,2024-11-26,Are0029,Are0012,Despachado Centralmente,GENERADOR,Solar
8,2026-01-19,P1D,2026-01-19,TPUY,PARQUE SOLAR TEPUY,EPMG,83000.0,2024-06-12,Are0022,Are0128,Despachado Centralmente,GENERADOR,Solar
9,2026-01-19,P1D,2026-01-19,4YCF,SHANGRI LA,ISGG,160000.0,2025-10-23,Are0031,Are0128,Despachado Centralmente,GENERADOR,Solar


In [ ]:
dataset_id = '7F18B1'   # Código del dataset que se desea consultar,
                        #  este código se obtiene del listado de variables, en la columna "DatasetID" y obtiene el Listado de plantas en pruebas
fecha_inicio = '2026-01-01'
fecha_fin = '2026-05-05'
simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
df_plantasPruebas = simem.main()
df_plantasPruebas.head(10)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Listado de plantas de generación en pruebas"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0
Extraccion de registros: 31.2107412815094
End of data extracting process
****************************************************************************************************


,FechaPublicacion,Fecha,CodigoPlanta,NombrePlanta,TipoGeneracion,CodigoSICAgente,CapacidadEfectivaNeta,CodigoSubAreaOperativa,CodigoAreaOperativa,TipoDespachoRecurso,TipoClasificacion
0,2026-01-22,2026-01-21,3J85,NUMBANA,Solar,ERRG,9900.0,Are0031,Are0128,No Despachado Centralmente,GENERADOR
1,2026-01-22,2026-01-21,5IW1,GD EMPERATRIZ I,Solar,SNCG,960.0,Are0021,Are0127,No Despachado Centralmente,GENERADOR DISTRIBUIDO
2,2026-01-22,2026-01-21,3PXJ,BUENAVISTA,Solar,GRCG,6800.0,Are0020,Are0127,No Despachado Centralmente,GENERADOR
3,2026-01-22,2026-01-21,5AXG,VALLEDUPAR III,Solar,EPSG,19900.0,Are0025,Are0127,No Despachado Centralmente,GENERADOR
4,2026-01-22,2026-01-21,3EXH,PARQUE EOLICO WESP01,Eolica,ISGG,12000.0,Are0025,Are0127,No Despachado Centralmente,GENERADOR
5,2026-01-22,2026-01-21,3QK6,AUTOG BIOS CIENAGA ORO,Solar,EPMG,1100.0,Are0021,Are0127,No Despachado Centralmente,AUTOGENERADOR
6,2026-01-22,2026-01-21,5BPX,GD AGUSTIN I,Solar,UNGG,990.0,Are0029,Are0012,No Despachado Centralmente,GENERADOR DISTRIBUIDO
7,2026-01-22,2026-01-21,5JDL,PARQUE SOLAR EL JARDIN,Solar,GNYG,9000.0,Are0025,Are0127,No Despachado Centralmente,GENERADOR
8,2026-01-22,2026-01-21,4UZM,SOLAR PALMASECA I,Solar,EPSG,13000.0,Are0032,Are0128,No Despachado Centralmente,GENERADOR
9,2026-01-22,2026-01-21,3GJQ,TERMOTASAJERO DOS SOLAR,Solar,TERG,4000.0,Are0029,Are0012,No Despachado Centralmente,GENERADOR


In [85]:
dataset_id = '841808' # Código del dataset que se desea consultar,  y permite obtener el listado de áreas operativas
fecha_inicio = '2026-01-01'
fecha_fin = '2026-05-05'
simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
df_areasoperativas = simem.main()
df_areasoperativas.head(10)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Listado de Áreas Operativas del Sistema de Interconectado Nacional"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0009989738464355469
Extraccion de registros: 13.683420181274414
End of data extracting process
****************************************************************************************************


,Fecha,CodigoAreaOperativa,NombreAreaOperativa,FechaEjecucion
0,2026-01-12,Are0065,Area Venezuela,2026-01-13
1,2026-01-12,Are0056,Area Oriental,2026-01-13
2,2026-01-12,Are0064,Area Ecuador,2026-01-13
3,2026-01-12,Are0127,Area Caribe,2026-01-13
4,2026-01-12,Are0012,Area Nordeste,2026-01-13
5,2026-01-12,Are0002,Area Antioquia,2026-01-13
6,2026-01-12,Are0128,Area Suroccidental,2026-01-13
7,2026-01-12,Are0010,Area SIN,2026-01-13
8,2026-01-10,Are0065,Area Venezuela,2026-01-11
9,2026-01-10,Are0056,Area Oriental,2026-01-11


In [86]:
# Renombrar columna en df_plantas
df_plantas.rename(columns={'CapEfectivaNeta': 'CapacidadEfectivaNeta', }, inplace=True)

# Columnas para el union all
cols = ['Fecha', 'CodigoPlanta', 'NombrePlanta', 'TipoGeneracion', 'CodigoSICAgente',
        'CapacidadEfectivaNeta', 'CodigoSubAreaOperativa', 'CodigoAreaOperativa',
        'TipoDespachoRecurso', 'TipoClasificacion']

# Union all solo con las columnas seleccionadas
df_union = pd.concat([df_plantas[cols], df_plantasPruebas[cols]], ignore_index=True)
df_union.head(10)

,Fecha,CodigoPlanta,NombrePlanta,TipoGeneracion,CodigoSICAgente,CapacidadEfectivaNeta,CodigoSubAreaOperativa,CodigoAreaOperativa,TipoDespachoRecurso,TipoClasificacion
0,2026-01-19,3IZ6,PARQUE SOLAR LA UNION,Solar,SLUG,100000.0,Are0021,Are0127,Despachado Centralmente,GENERADOR
1,2026-01-19,GYPO,GUAYEPO,Solar,ENDG,370000.0,Are0016,Are0127,Despachado Centralmente,GENERADOR
2,2026-01-19,3INX,CARACOLI I,Solar,SDCG,50000.0,Are0016,Are0127,Despachado Centralmente,GENERADOR
3,2026-01-19,3HF5,FUNDACION,Solar,ENDG,100000.0,Are0025,Are0127,Despachado Centralmente,GENERADOR
4,2026-01-19,EPFV,EL PASO,Solar,ENDG,68000.0,Are0025,Are0127,Despachado Centralmente,GENERADOR
5,2026-01-19,3DDT,LATAM SOLAR LA LOMA,Solar,ENDG,150000.0,Are0025,Are0127,Despachado Centralmente,GENERADOR
6,2026-01-19,MATA,LA MATA,Solar,SLMG,80000.0,Are0029,Are0012,Despachado Centralmente,GENERADOR
7,2026-01-19,3IQA,SUNNORTE,Solar,GNSG,35000.0,Are0029,Are0012,Despachado Centralmente,GENERADOR
8,2026-01-19,TPUY,PARQUE SOLAR TEPUY,Solar,EPMG,83000.0,Are0022,Are0128,Despachado Centralmente,GENERADOR
9,2026-01-19,4YCF,SHANGRI LA,Solar,ISGG,160000.0,Are0031,Are0128,Despachado Centralmente,GENERADOR


In [124]:
# Agregar columna Fecha (solo año-mes-día) a data_ultima_version
data_ultima_version['Fecha'] = pd.to_datetime(data_ultima_version['FechaHora']).dt.normalize()

# Asegurar que Fecha en df_union también sea datetime
df_union['Fecha'] = pd.to_datetime(df_union['Fecha'])

# Merge entre data_ultima_version y df_union por Fecha y CodigoPlanta
df_generacionCompleto = data_ultima_version.merge(df_union, on=['Fecha', 'CodigoPlanta'], how='left')
df_generacionCompleto.head(10)

,FechaHora,Version,CodigoPlanta,CodigoVariable,CodigoDuracion,UnidadMedida,CodigoSICAgente_x,Valor,Fecha,NombrePlanta,TipoGeneracion,CodigoSICAgente_y,CapacidadEfectivaNeta,CodigoSubAreaOperativa,CodigoAreaOperativa,TipoDespachoRecurso,TipoClasificacion
0,2026-01-26T09:00:00,TX3,TSR1,GReal,PT1H,kWh,EPMG,0.00,2026-01-26,TERMOSIERRA CC,Termica,EPMG,352000.0,Are0015,Are0002,Despachado Centralmente,CICLO COMBINADO
1,2026-01-26T04:00:00,TX3,COE1,GReal,PT1H,kWh,ENGG,1003.95,2026-01-26,COELLO,Hidraulica,ENGG,1200.0,Are0031,Are0128,No Despachado Centralmente,GENERADOR
2,2026-01-26T10:00:00,TX3,3EV3,GReal,PT1H,kWh,EPSG,8513.30,2026-01-26,LA MEDINA,Solar,EPSG,9900.0,Are0031,Are0128,No Despachado Centralmente,GENERADOR
3,2026-01-26T13:00:00,TX3,TBQ4,GReal,PT1H,kWh,TBSG,0.00,2026-01-26,BARRANQUILLA 4,Termica,TBSG,60000.0,Are0016,Are0127,Despachado Centralmente,GENERADOR
4,2026-01-26T21:00:00,TX3,3HG9,GReal,PT1H,kWh,ERRG,0.00,2026-01-26,COMUNIDAD EL SALVADOR II,Solar,ERRG,3.7,Are0015,Are0002,No Despachado Centralmente,GENERADOR DISTRIBUIDO
5,2026-01-26T00:00:00,TX3,3GPZ,GReal,PT1H,kWh,EPSG,0.00,2026-01-26,MONTELIBANO,Solar,EPSG,9900.0,Are0020,Are0127,No Despachado Centralmente,GENERADOR
6,2026-01-26T03:00:00,TX3,GTPE,GReal,PT1H,kWh,EPMG,556560.00,2026-01-26,GUATAPE,Hidraulica,EPMG,560000.0,Are0015,Are0002,Despachado Centralmente,GENERADOR
7,2026-01-26T14:00:00,TX3,PTC1,GReal,PT1H,kWh,FERG,1101.00,2026-01-26,PATICO - LA CABRERA,Hidraulica,FERG,1480.0,Are0019,Are0128,No Despachado Centralmente,GENERADOR
8,2026-01-26T16:00:00,TX3,TR2G,GReal,PT1H,kWh,ISGG,11114.28,2026-01-26,TRINA-VATIA BSLII,Solar,ISGG,19900.0,Are0027,Are0056,No Despachado Centralmente,GENERADOR
9,2026-01-26T02:00:00,TX3,ALBG,GReal,PT1H,kWh,EPSG,11906.00,2026-01-26,ALBAN,Hidraulica,EPSG,427000.0,Are0032,Are0128,Despachado Centralmente,GENERADOR


In [126]:
# Eliminar registros con Fecha entre 2026-04-27 y 2026-04-30 y Version == 'TX2'
mask = (
    (df_generacionCompleto['Fecha'] >= '2026-04-27') &
    (df_generacionCompleto['Fecha'] <= '2026-04-30') &
    (df_generacionCompleto['Version'] == 'TX2')
)
print(f"Registros a eliminar: {mask.sum()}")
df_generacionCompleto = df_generacionCompleto[~mask]
print(f"Registros restantes: {len(df_generacionCompleto)}")

Registros a eliminar: 53040
Registros restantes: 1556760


In [127]:
dataaa =  df_generacionCompleto[df_generacionCompleto['Fecha']==pd.to_datetime('2026-04-27')]
dataaa.to_clipboard()

### Análisis de Generación Distribuida (Ene - May 2026)

Se filtra el dataframe por `TipoClasificacion == 'GENERADOR DISTRIBUIDO'` y se realizan los siguientes ejercicios:

1. **Generación total mensual por Tipo de Generación** (gráfica de barras)
2. **Generación total por Área Operativa** (gráfica de barras horizontal)
3. **Top 10 Agentes por generación total** (gráfica de barras)
4. **Evolución diaria de generación por Tipo de Generación** (gráfica de líneas)
5. **Distribución de generación por Área Operativa y Tipo de Generación** (gráfica de barras apiladas)

In [128]:
# Filtrar solo Generadores Distribuidos
df_gd = df_generacionCompleto[df_generacionCompleto['TipoClasificacion'] == 'GENERADOR DISTRIBUIDO'].copy()
df_gd['Mes'] = df_gd['Fecha'].dt.to_period('M').astype(str)
print(f"Registros de Generador Distribuido: {len(df_gd)}")
print(f"Tipos de Generación: {df_gd['TipoGeneracion'].unique()}")
print(f"Áreas Operativas: {df_gd['CodigoAreaOperativa'].nunique()}")
print(f"Agentes (CodigoSICAgente): {df_gd['CodigoSICAgente_x'].nunique()}")

Registros de Generador Distribuido: 481320
Tipos de Generación: ['Solar' 'Hidraulica']
Áreas Operativas: 5
Agentes (CodigoSICAgente): 34


In [129]:
# Obtener mapeo único de CodigoAreaOperativa -> NombreAreaOperativa
areas_unicas = df_areasoperativas[['CodigoAreaOperativa', 'NombreAreaOperativa']].drop_duplicates()

# Join para agregar NombreAreaOperativa a df_gd
df_gd = df_gd.merge(areas_unicas, on='CodigoAreaOperativa', how='left')
print(f"Columnas disponibles: {list(df_gd.columns)}")
print(f"Áreas con nombre: {df_gd['NombreAreaOperativa'].nunique()}")

Columnas disponibles: ['FechaHora', 'Version', 'CodigoPlanta', 'CodigoVariable', 'CodigoDuracion', 'UnidadMedida', 'CodigoSICAgente_x', 'Valor', 'Fecha', 'NombrePlanta', 'TipoGeneracion', 'CodigoSICAgente_y', 'CapacidadEfectivaNeta', 'CodigoSubAreaOperativa', 'CodigoAreaOperativa', 'TipoDespachoRecurso', 'TipoClasificacion', 'Mes', 'NombreAreaOperativa']
Áreas con nombre: 5


#### 1. Generación total mensual por Tipo de Generación

In [130]:
print(list(df_areasoperativas.columns))
print(list(df_areasoperativas.columns))
print(df_areasoperativas.head(3))
print(list(df_areasoperativas.columns))# Generación mensual por Tipo de Generación
gen_mensual_tipo = df_gd.groupby(['Mes', 'TipoGeneracion'])['Valor'].sum().reset_index()

fig1 = go.Figure()
for tipo in gen_mensual_tipo['TipoGeneracion'].unique():
    datos = gen_mensual_tipo[gen_mensual_tipo['TipoGeneracion'] == tipo]
    fig1.add_trace(go.Bar(x=datos['Mes'], y=datos['Valor'], name=tipo))

fig1.update_layout(
    title='Generación Total Mensual por Tipo de Generación (Generadores Distribuidos)',
    xaxis_title='Mes',
    yaxis_title='Generación (kWh)',
    barmode='group',
    template='plotly_white'
)
fig1.show()

['Fecha', 'CodigoAreaOperativa', 'NombreAreaOperativa', 'FechaEjecucion']
['Fecha', 'CodigoAreaOperativa', 'NombreAreaOperativa', 'FechaEjecucion']
        Fecha CodigoAreaOperativa NombreAreaOperativa FechaEjecucion
0  2026-01-12             Are0065      Area Venezuela     2026-01-13
1  2026-01-12             Are0056       Area Oriental     2026-01-13
2  2026-01-12             Are0064        Area Ecuador     2026-01-13
['Fecha', 'CodigoAreaOperativa', 'NombreAreaOperativa', 'FechaEjecucion']


#### 2. Generación total por Área Operativa

In [131]:
# Generación total por Área Operativa
gen_area = df_gd.groupby('NombreAreaOperativa')['Valor'].sum().reset_index()
gen_area = gen_area.sort_values('Valor', ascending=True)

fig2 = go.Figure(go.Bar(
    x=gen_area['Valor'],
    y=gen_area['NombreAreaOperativa'],
    orientation='h'
))
fig2.update_layout(
    title='Generación Total por Área Operativa (Ene-May 2026)',
    xaxis_title='Generación (kWh)',
    yaxis_title='Área Operativa',
    template='plotly_white',
    height=500
)
fig2.show()

#### 3. Top 10 Agentes por Generación Total

In [132]:
dataset_id = '972263' # Código del dataset que se desea consultar,  y permite obtener el listado de agentes
fecha_inicio = '2026-01-01'
fecha_fin = '2026-05-05'
simem = ReadSIMEM(dataset_id, fecha_inicio, fecha_fin)
df_agentes = simem.main()           
df_agentes.head(10)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Listado de agentes registrados"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0
Extraccion de registros: 17.150083541870117
End of data extracting process
****************************************************************************************************


,Fecha,CodigoDuracion,CodigoSICAgente,NombreAgente,ActividadAgente
0,2026-01-02,P1D,EMSC,ELECTRIFICADORA DEL META S.A. E.S.P.,Comercializador
1,2026-01-02,P1D,BIAC,BIA ENERGY S.A.S. E.S.P,Comercializador
2,2026-01-02,P1D,TRFG,TRINA SOLAR GENERADOR COLOMBIA,Generador
3,2026-01-02,P1D,LTEC,LATINOAMERICAN ENERGY S.A.S E.S.P.,Comercializador
4,2026-01-02,P1D,OROG,PARQUE SOLAR PUERTA DE ORO S.A.S. E.S.P.,Generador
5,2026-01-02,P1D,ORQG,LA ORQUIDEA SOLAR S.A.S. E.S.P.,Generador
6,2026-01-02,P1D,FYRG,FOTOVOLTAICO EL YARUMO SAS ESP,Generador
7,2026-01-02,P1D,PTTC,SOCIEDAD DE ECONOMIA MIXTA ALUMBRADO PUBLICO D...,Comercializador
8,2026-01-02,P1D,CTEC,COENERGIA S.A.S. E.S.P.,Comercializador
9,2026-01-02,P1D,MARG,SOLAR LAS MARIAS SAS ESP,Generador


In [133]:
# Top 10 Agentes con mayor generación
gen_agente = df_gd.groupby('CodigoSICAgente_x')['Valor'].sum().reset_index()
gen_agente = gen_agente.sort_values('Valor', ascending=False).head(10)

# Merge con df_agentes para obtener NombreAgente
agentes_unicos = df_agentes[['CodigoSICAgente', 'NombreAgente']].drop_duplicates()
gen_agente = gen_agente.merge(agentes_unicos, left_on='CodigoSICAgente_x', right_on='CodigoSICAgente', how='left')

In [134]:
fig3 = go.Figure(go.Bar(
    x=gen_agente['NombreAgente'],
    y=gen_agente['Valor'],
    marker_color='indianred'
))
fig3.update_layout(
    title='Top 10 Agentes por Generación Total (Generadores Distribuidos, Ene-May 2026)',
    xaxis_title='Agente',
    yaxis_title='Generación (kWh)',
    template='plotly_white'
)
fig3.show()

#### 4. Evolución diaria de generación por Tipo de Generación

In [135]:
# Evolución diaria de generación por Tipo de Generación
gen_diaria_tipo = df_gd.groupby(['Fecha', 'TipoGeneracion'])['Valor'].sum().reset_index()
gen_diaria_tipo.sort_values('Fecha', inplace=True)

fig4 = go.Figure()
for tipo in gen_diaria_tipo['TipoGeneracion'].unique():
    datos = gen_diaria_tipo[gen_diaria_tipo['TipoGeneracion'] == tipo]
    fig4.add_trace(go.Scatter(x=datos['Fecha'], y=datos['Valor'], mode='lines', name=tipo))

fig4.update_layout(
    title='Evolución Diaria de Generación por Tipo (Generadores Distribuidos)',
    xaxis_title='Fecha',
    yaxis_title='Generación (kWh)',
    xaxis={'tickformat': '%d-%b-%Y'},
    template='plotly_white'
)
fig4.show()

#### 5. Distribución de generación por Área Operativa y Tipo de Generación (barras apiladas)

In [136]:
# Distribución por Área Operativa y Tipo de Generación
gen_area_tipo = df_gd.groupby(['NombreAreaOperativa', 'TipoGeneracion'])['Valor'].sum().reset_index()

fig5 = go.Figure()
for tipo in gen_area_tipo['TipoGeneracion'].unique():
    datos = gen_area_tipo[gen_area_tipo['TipoGeneracion'] == tipo]
    fig5.add_trace(go.Bar(x=datos['NombreAreaOperativa'], y=datos['Valor'], name=tipo))

fig5.update_layout(
    title='Generación por Área Operativa y Tipo de Generación (Ene-May 2026)',
    xaxis_title='Área Operativa',
    yaxis_title='Generación (kWh)',
    barmode='stack',
    template='plotly_white'
)
fig5.show()

### Participación de la Generación Distribuida en el Total del Sistema

Se compara la generación distribuida (`TipoClasificacion == 'GENERADOR DISTRIBUIDO'`) contra la generación total del sistema (sin filtros) para evaluar su participación.

1. **Evolución diaria**: Generación total del sistema vs Generación distribuida
2. **Participación diaria (%)**: Porcentaje que representa la generación distribuida respecto al total
3. **Resumen mensual**: Tabla con totales y porcentaje de participación por mes
4. **Participación mensual (gráfico combinado)**: Barras para valores absolutos + línea para porcentaje

In [137]:
# Generación total del sistema por día (sin filtros)
gen_total_diaria = df_generacionCompleto.groupby('Fecha')['Valor'].sum().reset_index()
gen_total_diaria.rename(columns={'Valor': 'GenTotal'}, inplace=True)

# Generación distribuida por día
gen_gd_diaria = df_gd.groupby('Fecha')['Valor'].sum().reset_index()
gen_gd_diaria.rename(columns={'Valor': 'GenDistribuida'}, inplace=True)

# Merge ambas series
comparacion_diaria = gen_total_diaria.merge(gen_gd_diaria, on='Fecha', how='left').fillna(0)
comparacion_diaria['PctParticipacion'] = (comparacion_diaria['GenDistribuida'] / comparacion_diaria['GenTotal']) * 100

# Gráfico 6: Evolución diaria - Generación Total vs Generación Distribuida
from plotly.subplots import make_subplots

fig6 = make_subplots(specs=[[{"secondary_y": True}]])
fig6.add_trace(
    go.Scatter(x=comparacion_diaria['Fecha'], y=comparacion_diaria['GenTotal'],
               name='Generación Total', line=dict(color='royalblue', width=2)),
    secondary_y=False
)
fig6.add_trace(
    go.Scatter(x=comparacion_diaria['Fecha'], y=comparacion_diaria['GenDistribuida'],
               name='Generación Distribuida', line=dict(color='orange', width=2)),
    secondary_y=True
)
fig6.update_layout(
    title='Evolución Diaria: Generación Total del Sistema vs Generación Distribuida (Ene-May 2026)',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig6.update_yaxes(title_text='Generación Total (kWh)', secondary_y=False)
fig6.update_yaxes(title_text='Generación Distribuida (kWh)', secondary_y=True)
fig6.show()

In [138]:
df_generacionCompleto.to_clipboard()

In [139]:
gen_total_diaria.to_clipboard()

In [140]:
# Gráfico 7: Participación diaria (%) de la Generación Distribuida
fig7 = go.Figure()
fig7.add_trace(go.Scatter(
    x=comparacion_diaria['Fecha'],
    y=comparacion_diaria['PctParticipacion'],
    mode='lines+markers',
    line=dict(color='green', width=2),
    marker=dict(size=4),
    name='% Participación GD'
))
fig7.add_hline(y=comparacion_diaria['PctParticipacion'].mean(),
               line_dash='dash', line_color='red',
               annotation_text=f"Promedio: {comparacion_diaria['PctParticipacion'].mean():.2f}%",
               annotation_position='top left')
fig7.update_layout(
    title='Participación Diaria de la Generación Distribuida en el Total del Sistema (Ene-May 2026)',
    xaxis_title='Fecha',
    yaxis_title='Participación (%)',
    template='plotly_white'
)
fig7.show()

In [141]:
# Tabla resumen mensual: Generación Total vs Generación Distribuida
comparacion_diaria['Mes'] = comparacion_diaria['Fecha'].dt.to_period('M').astype(str)

resumen_mensual = comparacion_diaria.groupby('Mes').agg(
    GenTotal_kWh=('GenTotal', 'sum'),
    GenDistribuida_kWh=('GenDistribuida', 'sum')
).reset_index()
resumen_mensual['Participacion_pct'] = (resumen_mensual['GenDistribuida_kWh'] / resumen_mensual['GenTotal_kWh'] * 100).round(3)

# Formatear valores en GWh para mejor lectura
resumen_mensual['GenTotal_GWh'] = (resumen_mensual['GenTotal_kWh'] / 1e6).round(2)
resumen_mensual['GenDistribuida_GWh'] = (resumen_mensual['GenDistribuida_kWh'] / 1e6).round(4)

resumen_mensual[['Mes', 'GenTotal_GWh', 'GenDistribuida_GWh', 'Participacion_pct']]

,Mes,GenTotal_GWh,GenDistribuida_GWh,Participacion_pct
0,2026-01,7278.00,19.3835,0.266
1,2026-02,6590.69,19.5462,0.297
2,2026-03,7457.05,25.7304,0.345
3,2026-04,7210.35,25.4253,0.353
4,2026-05,686.95,2.3474,0.342


In [142]:
# Gráfico 8: Participación mensual - Barras (GWh) + Línea (%)
fig8 = make_subplots(specs=[[{"secondary_y": True}]])

fig8.add_trace(
    go.Bar(x=resumen_mensual['Mes'], y=resumen_mensual['GenTotal_GWh'],
           name='Generación Total (GWh)', marker_color='royalblue', opacity=0.7),
    secondary_y=False
)
fig8.add_trace(
    go.Bar(x=resumen_mensual['Mes'], y=resumen_mensual['GenDistribuida_GWh'],
           name='Generación Distribuida (GWh)', marker_color='orange', opacity=0.9),
    secondary_y=False
)
fig8.add_trace(
    go.Scatter(x=resumen_mensual['Mes'], y=resumen_mensual['Participacion_pct'],
               name='% Participación', mode='lines+markers+text',
               text=resumen_mensual['Participacion_pct'].apply(lambda x: f"{x:.3f}%"),
               textposition='top center',
               line=dict(color='red', width=3), marker=dict(size=8)),
    secondary_y=True
)

fig8.update_layout(
    title='Participación Mensual de la Generación Distribuida en el Total del Sistema (Ene-May 2026)',
    barmode='group',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig8.update_yaxes(title_text='Generación (GWh)', secondary_y=False)
fig8.update_yaxes(title_text='Participación (%)', secondary_y=True)
fig8.show()